In [36]:
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import classification_report, multilabel_confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import copy

# OpenMP 충돌 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [37]:
Config = {
    "NUM_CLASSES": 5,         # 0 normal, 1 dos, 2 fuzzing, 3 replay, 4 spoofing
    "BATCH_SIZE": 64,
}

In [38]:
# ================================
# 1. Causal Convolution 레이어 정의
# ================================

class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )
        self.left_pad = (kernel_size - 1) * dilation

    def forward(self, x):
        x = F.pad(x, (self.left_pad, 0))  # 왼쪽만 패딩
        return super().forward(x)


In [39]:
m = weight_norm(nn.Linear(20,40), name = 'weight')
m
m.weight_g.size()
m.weight_v.size()

c:\Users\user\anaconda3\envs\ids_masters\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


torch.Size([40, 20])

In [40]:
class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernal_size=3,  dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernal_size-1)*dilation
        self.conv1 = weight_norm(CausalConv1d(n_inputs, n_outputs, kernel_size=kernal_size, dilation=dilation))
        
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation))
        
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
      
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()
    def init_weights(self):
        def init_one(layer):
            # weight_norm이면 weight_orig가 진짜 파라미터
            w = getattr(layer, "weight_orig", None)
            if w is None:
                w = layer.weight
            nn.init.kaiming_normal_(w)

            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

        # conv1, conv2가 무엇이든(래퍼든 상속이든) 일단 'Conv1d 파라미터 가진 최종 모듈'에 적용
        init_one(self.conv1)
        init_one(self.conv2)

        if self.downsample is not None:
            nn.init.kaiming_normal_(self.downsample.weight)
            if self.downsample.bias is not None:
                nn.init.zeros_(self.downsample.bias)
    def forward(self, x):
        # x: (B, C, L)
        out = self.conv1(x)          # CausalConv1d 안에서 패딩 + 오른쪽 잘라내기
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        res = x if self.downsample is None else self.downsample(x)
        # CausalConv1d가 길이를 유지하니까 따로 slice 안 해도 됨
        return self.relu(out + res)

In [41]:
class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channel, kernel_size=2, dropout =0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channel)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channel[i-1]
            out_channels = num_channel[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size,  dilation = dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

In [42]:
# class DualStageAttention(nn.Module):
#     def __init__(self, num_features, reduction=4):
#         super(DualStageAttention, self).__init__()
#         # 64채널을 처리할 수 있도록 설계
#         hidden = max(num_features // reduction, 16) 
        
#         # Feature(Channel) Attention
#         self.feature_mlp = nn.Sequential(
#             nn.Linear(num_features * 2, hidden),
#             nn.ReLU(),
#             nn.Linear(hidden, num_features),
#             nn.Sigmoid()
#         )
        
#         # Time-wise Attention (안정적인 점수 계산을 위해 Tanh 제거)
#         self.time_fc = nn.Sequential(
#             nn.Linear(num_features, hidden),
#             nn.ReLU(),
#             nn.Linear(hidden, 1)
#         )

#     def forward(self, x):
#         # x shape: (B, T, F) -> 여기서는 (Batch, 64, 64)
#         identity = x 
#         b, t, f = x.size()

#         # [Feature Stage]
#         f_avg = torch.mean(x, dim=1)
#         f_max, _ = torch.max(x, dim=1)
#         f_cat = torch.cat([f_avg, f_max], dim=1)
#         attn_feat = self.feature_mlp(f_cat) # (B, F)
#         x_feat_weighted = x * attn_feat.unsqueeze(1)

#         # [Time Stage]
#         time_score = self.time_fc(x_feat_weighted).squeeze(-1) # (B, T)
#         attn_time = F.softmax(time_score, dim=1)
#         x_time_weighted = x_feat_weighted * attn_time.unsqueeze(-1)

#         # [Residual Connection] 중요: 성능 하락 방지 장치
#         # 어텐션이 적용된 값에 원본을 더해 특징 소실을 막음
#         x_final = x_time_weighted + identity 
        
#         return x_final, attn_feat, attn_time

In [43]:
class SeqIDS(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.5):
        super(SeqIDS, self).__init__()

        # 1) TCN backbone: 입력 9채널 → [32, 64] 채널
        self.tcn = TemporalConvNet(
            num_inputs=9,          # feature 개수
            num_channel=[32, 64],  # 각 레벨 채널 수
            kernel_size=3,
            dropout=dropout_rate
        )

        # 2) 64채널에 대한 이중 어텐션
        #self.attention = DualStageAttention(num_features=64)

        # 3) 분류기
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Conv1d(64, num_classes, kernel_size=1)

    def forward(self, x, return_attn=False):
        # x: (B, 9, L)  <- 현재 인코딩이 (B, 9, 64)일 거임

        # 1. TCN으로 시계열 특징 추출
        x = self.tcn(x)          # (B, 64, L)

        # #2. 어텐션 적용 (B, T, F)로 transpose
        # x_t = x.permute(0, 2, 1)  # (B, L, 64)
        # x_weighted, attn_feat, attn_time = self.attention(x_t)
        # x = x_weighted.permute(0, 2, 1)  # (B, 64, L)

        # 3. 분류
        x = self.dropout(x)
        logits = self.classifier(x)      # (B, num_classes, L)

        if return_attn:
            return logits
        return logits

In [44]:
# ================================
# 3. 데이터셋 클래스
# ================================
class SeqWindowDataset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        # [수정] NumPy 배열이 들어올 경우를 대비한 변환 로직
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        # [추가] NaN(결측치)을 0.0으로 안전하게 대체하여 학습 붕괴 방지
        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [45]:
# ================================
# 4. 평가 함수 (Window Level Evaluation)
# ================================
def evaluate_packet_level_multilabel(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    # 라벨 이름 (인코딩 시 설정한 순서대로)
    target_names = ['Normal', 'DoS', 'Fuzzing', 'Replay', 'Spoofing']
    mlb = MultiLabelBinarizer(classes=[0, 1, 2, 3, 4])

    print("\n🔍 평가 진행 중...")
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            
            # (Batch, 64) - 각 패킷별 예측 클래스
            preds = logits.argmax(dim=1)

            # 2. 모든 패킷을 1차원으로 펼침 (Flatten)
            # numpy를 사용하면 리스트 comprehension보다 훨씬 빠릅니다.
            p_flat = preds.cpu().numpy().flatten()
            t_flat = y_batch.cpu().numpy().flatten()
            
            # Multi-label 평가 포맷에 맞춰 리스트화
            all_preds.extend([[p] for p in p_flat])
            all_targets.extend([[t] for t in t_flat])

    # 3. 이진 변환 및 지표 계산
    y_true = mlb.fit_transform(all_targets)
    y_pred = mlb.transform(all_preds)

    print("\n" + "="*60)
    print("📊 [Packet-Level] Multi-Label Evaluation Results")
    print("   (Total Packets Evaluated: {:,})".format(len(all_targets)))
    print("="*60)
    # 이제 Support 숫자가 윈도우 개수가 아닌 전체 패킷 개수로 나올 것입니다.
    print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
    
    return multilabel_confusion_matrix(y_true, y_pred)

In [46]:
# ================================
# 5. 학습 함수 (일반화 5대장 적용 완료)
# ================================
def train_seqids_advanced(dataset_npz_path, epochs, lr=1e-4, device=None, patience=3):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥 실행 디바이스: {device}")

    # 1. 데이터 로드
    print(f"📂 데이터 로딩 중: {dataset_npz_path}")
    data_np = np.load(dataset_npz_path)
    X_np, y_np = data_np["X"], data_np["y"]

    # NaN이 X의 어느 feature 중에서 가장 많이 나오는지 확인(인코딩 잘못됐는지 확인)
    for i in range(X_np.shape[1]):
        feat_nan = np.isnan(X_np[:, i, :]).sum()
        print(f"Feature {i}의 NaN 개수: {feat_nan}")

    
    nan_count_X = np.isnan(X_np).sum()
    nan_count_y = np.isnan(y_np).sum()
    print(f"🔍 데이터 검사 결과:")
    print(f"   - X 내 NaN 개수: {nan_count_X}개")
    print(f"   - y 내 NaN 개수: {nan_count_y}개")
    if nan_count_X > 0:
        print("   ⚠️ 주의: 입력 데이터(X)의 NaN은 0.0으로 자동 대체되어 학습됩니다.")


    inf_count = np.isinf(X_np).sum() # 무한대 체크 추가
    print(f"🔍 X 내 inf 개수: {inf_count}개")
    if inf_count > 0:
        print("⚠️ 경고: 데이터에 inf(무한대)가 포함되어 있습니다. 인코딩 스크립트를 수정하세요.")


    full_dataset = SeqWindowDataset(X_np, y_np)

    # [일반화 1] Validation Split (80:20)
    total_size = len(full_dataset)
    train_size = int(0.8 * total_size)
    val_size = total_size - train_size
    
    # 시드 고정 (재현성을 위해)
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)
    
    print(f"📊 데이터 분할 완료: 학습 {train_size}개 / 검증 {val_size}개")
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    # 2. 모델 및 옵티마이저
    # [일반화 2] Dropout 적용 (0.5)
    model = SeqIDS(num_classes=5, dropout_rate=0.5).to(device)
    
    # [일반화 3] Weight Decay (L2 Regularization) 적용 -> 1e-4
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-2)
    # [일반화 4] Learning Rate Scheduler (성능 정체 시 학습률 감소)
    # 3번(patience) 동안 Val Loss가 안 줄어들면 학습률을 반토막(0.5) 냄
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    
    # 불균형 데이터 가중치 (공격 클래스 중요도 Up)
    weights = torch.tensor([1.0, 2.0, 2.0, 2.0, 2.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # [일반화 5] Early Stopping 변수
    best_val_loss = float('inf')
    best_model_state = None
    patience_check = 0

    print("\n🚀 학습 시작 (Advanced Mode)...")
    
    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            logits = model(X_batch)
            # (Batch, Class, Time) -> (Batch*Time, Class) 형태로 변형하여 Loss 계산
            loss = criterion(logits.permute(0, 2, 1).reshape(-1, 5), y_batch.reshape(-1))
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)

        # --- Validation ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                
                logits = model(X_val)
                loss = criterion(logits.permute(0, 2, 1).reshape(-1, 5), y_val.reshape(-1))
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        
        # 스케줄러 업데이트 (Val Loss 기준)
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch}/{epochs}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")

        # --- Early Stopping Logic ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict()) # 모델 가중치 백업
            patience_check = 0 # 카운트 초기화
            # print("  ✅ Best Model 갱신!")
        else:
            patience_check += 1
            print(f"  ⚠️ 검증 손실 개선 안됨 (Patience: {patience_check}/{patience})")
            if patience_check >= patience:
                print(f"🛑 Early Stopping 발동! 학습을 조기 종료합니다.")
                break

    # 학습 종료 후, 가장 좋았던 모델 상태로 복구
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("\n💾 검증 성능이 가장 좋았던(Best) 모델로 복구되었습니다.")

    # 검증 로더도 반환하여 최종 평가에 사용
    return model, val_loader, device

In [47]:
# ================================
# 6. 메인 실행 블록
# ================================
if __name__ == "__main__":
    # [주의] 본인의 실제 파일 경로로 수정하세요
    DATA_PATH = "C:/Users/user/Desktop/IDS_masters/dataset/challenge_training_dataset.npz"
    
    # 1) 향상된 학습 진행 (Validation, Dropout, BatchNorm, Scheduler 등 모두 적용)
    # 리턴받는 val_loader는 학습에 쓰지 않은 '순수 검증용' 데이터입니다.
    trained_model, val_loader, device = train_seqids_advanced(DATA_PATH, epochs=20, patience=7)

    # 2) 검증 데이터셋(20%)에 대한 최종 상세 평가
    print("\n🔍 최종 검증 데이터셋(Validation Set) 평가 결과:")
    evaluate_packet_level_multilabel(trained_model, val_loader, device)

    # 3) 모델 저장
    MODEL_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/model/TCN.pth"
    torch.save(trained_model.state_dict(), MODEL_SAVE_PATH)
    print(f"\n💾 Best 모델 저장이 완료되었습니다: {MODEL_SAVE_PATH}")

🖥 실행 디바이스: cuda
📂 데이터 로딩 중: C:/Users/user/Desktop/IDS_masters/dataset/challenge_training_dataset.npz
Feature 0의 NaN 개수: 0
Feature 1의 NaN 개수: 0
Feature 2의 NaN 개수: 0
Feature 3의 NaN 개수: 0
Feature 4의 NaN 개수: 0
Feature 5의 NaN 개수: 0
Feature 6의 NaN 개수: 0
Feature 7의 NaN 개수: 0
Feature 8의 NaN 개수: 0
🔍 데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
🔍 X 내 inf 개수: 0개
📊 데이터 분할 완료: 학습 93800개 / 검증 23450개

🚀 학습 시작 (Advanced Mode)...
Epoch [1/20] Train Loss: 0.42962 | Val Loss: 0.20814 | LR: 0.000100
Epoch [2/20] Train Loss: 0.21996 | Val Loss: 0.17945 | LR: 0.000100
Epoch [3/20] Train Loss: 0.19914 | Val Loss: 0.17299 | LR: 0.000100
Epoch [4/20] Train Loss: 0.19354 | Val Loss: 0.17145 | LR: 0.000100
Epoch [5/20] Train Loss: 0.18885 | Val Loss: 0.16910 | LR: 0.000100
Epoch [6/20] Train Loss: 0.18382 | Val Loss: 0.16633 | LR: 0.000100
Epoch [7/20] Train Loss: 0.18032 | Val Loss: 0.16230 | LR: 0.000100
Epoch [8/20] Train Loss: 0.17827 | Val Loss: 0.16098 | LR: 0.000100
Epoch [9/20] Train Loss: 0.17668 

In [48]:
# ==========================================
# 설정 (Configuration)
# ==========================================
# [수정] 인코딩 코드로 만든 .pt 파일 경로
TEST_DATA_PATH = "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_dataset.npz"
# [수정] 학습된 모델 가중치 파일 경로
MODEL_PATH = "C:/Users/user/Desktop/IDS_masters/model/TCN.pth"

BATCH_SIZE = 1024
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 라벨 이름 (결과 리포트용)
LABEL_NAME = {0: "Normal", 1: "DoS", 2: "Fuzzing", 3: "Replay", 4: "Spoofing"}
TARGET_NAMES = [LABEL_NAME[i] for i in range(5)]

In [49]:
#################################
#  6. Testing
##################################
data_np = np.load(TEST_DATA_PATH)
X_np, y_np = data_np["X"], data_np["y"]

test_ds = SeqWindowDataset(X_np, y_np)

test_loader = DataLoader(
    test_ds,
    batch_size=Config["BATCH_SIZE"],
    shuffle=False
)

model = SeqIDS().to(device)

# 저장된 weight 로드
state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)

### Start Evolution ###
model.eval()

correct = 0
total = 0
num_classes = 5

conf_mat = torch.zeros(num_classes, num_classes, dtype=torch.int64)

with torch.no_grad():
    for input, labels in test_loader:
        input = input.to(device)
        labels = labels.to(device)

        logits = model(input)

        pred = logits.argmax(dim=1)

        total += labels.size(0)
        correct += (pred == labels).sum().item()

        # confusion matrix 누적 (CPU에서 int로)
        for t, p in zip(labels.view(-1).cpu(), pred.view(-1).cpu()):
            conf_mat[t.long(), p.long()] += 1

accuracy = correct / total

row_sum = conf_mat.sum(dim=1)
tp = conf_mat.diag()
fp = conf_mat.sum(dim=0) - tp
fn = row_sum - tp

precision_per_class = tp / (tp + fp + 1e-12)
recall_per_class    = tp / (tp + fn + 1e-12)
f1_per_class        = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + 1e-12)

present = row_sum > 0
precision_macro = precision_per_class[present].mean().item()
recall_macro    = recall_per_class[present].mean().item()
f1_macro        = f1_per_class[present].mean().item()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision(macro, present only): {precision_macro:.4f}")
print(f"Recall(macro, present only)   : {recall_macro:.4f}")
print(f"F1(macro, present only)       : {f1_macro:.4f}")
print("Confusion Matrix:")
print(conf_mat)


EVAL_CLASSES = [0, 1, 2, 4]
eval_idx = torch.tensor(EVAL_CLASSES)

precision_macro = precision_per_class[eval_idx].mean().item()
recall_macro    = recall_per_class[eval_idx].mean().item()
f1_macro        = f1_per_class[eval_idx].mean().item()

LABEL_NAME = {0:"Normal",1:"Dos",2:"Fuzzing",4:"Spoofing"}

print("\n=== Per-class (Attack) Performance ===")
for i in EVAL_CLASSES:
    total_i = int(row_sum[i].item())
    correct_i = int(tp[i].item())
    acc_i = 100.0 * correct_i / total_i if total_i > 0 else 0.0
    print(f"{LABEL_NAME[i]:>10s} : {acc_i:6.2f}%  (correct {correct_i}/{total_i})")

c:\Users\user\anaconda3\envs\ids_masters\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Accuracy : 17.3553
Precision(macro, present only): 0.4096
Recall(macro, present only)   : 0.5708
F1(macro, present only)       : 0.3148
Confusion Matrix:
tensor([[ 7229406, 11958052,   752153,  2679021,  7834622],
        [     192,  1163149,        0,        0,    11701],
        [      93,     3003,   979563,     1035,        0],
        [       0,        0,        0,        0,        0],
        [   19556,  2056004,    63746,   214375,   150617]])

=== Per-class (Attack) Performance ===
    Normal :  23.74%  (correct 7229406/30453254)
       Dos :  98.99%  (correct 1163149/1175042)
   Fuzzing :  99.58%  (correct 979563/983694)
  Spoofing :   6.01%  (correct 150617/2504298)
